# BTL02 T0304 — RQ2: pretrained vs scratch và input size

Notebook chạy tuần tự ma trận 9 cấu hình YAML trên Colab GPU. Chọn runtime T4/A100 (tối thiểu GPU CUDA), thêm `KAGGLE_API_TOKEN` vào Colab Secrets nếu FER2013 chưa có trên Drive. Checkpoint, MLflow file store, CSV 24 cột và biểu đồ được lưu trên Drive. Không ghi token vào notebook/repository. Các metrics chỉ là kết quả thực nghiệm sau khi cell hoàn tất; notebook không chứa số liệu mẫu giả.


In [ ]:
from google.colab import drive
from pathlib import Path
drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive/BTL02/T0304_RQ2')
MLRUNS_DIR = DRIVE_ROOT / 'mlruns'
CHECKPOINTS_DIR = DRIVE_ROOT / 'checkpoints'
LOGS_DIR = DRIVE_ROOT / 'logs'
FIGURES_DIR = DRIVE_ROOT / 'figures'
for folder in (MLRUNS_DIR, CHECKPOINTS_DIR, LOGS_DIR, FIGURES_DIR): folder.mkdir(parents=True, exist_ok=True)
print('Artifacts persist on Drive:', DRIVE_ROOT)


In [ ]:
import os, subprocess, tempfile
from pathlib import Path
from google.colab import userdata
REPO_URL = 'https://github.com/Duy-DS/Deep_Learning_Group.git'
BRANCH = 'exp/BTL02_T0304_rq2_transfer_vs_scratch'
REPO_DIR = Path('/content/Deep_Learning_Group')
token = userdata.get('GITHUB_TOKEN')
git_env = os.environ.copy()
askpass_path = None
if token:
    askpass = '#!/bin/sh\ncase "$1" in *Username*) echo x-access-token ;; *) echo "$COLAB_GITHUB_TOKEN" ;; esac\n'
    fd, askpass_path = tempfile.mkstemp(prefix='colab-git-askpass-')
    with os.fdopen(fd, 'w', encoding='utf-8') as stream: stream.write(askpass)
    os.chmod(askpass_path, 0o700)
    git_env.update({'GIT_ASKPASS': askpass_path, 'GIT_TERMINAL_PROMPT': '0', 'COLAB_GITHUB_TOKEN': token})
try:
    if REPO_DIR.exists():
        subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', 'origin', BRANCH], check=True, env=git_env, capture_output=True, text=True)
        subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', BRANCH], check=True)
        subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only', 'origin', BRANCH], check=True, env=git_env, capture_output=True, text=True)
    else:
        subprocess.run(['git', 'clone', '--branch', BRANCH, '--single-branch', REPO_URL, str(REPO_DIR)], check=True, env=git_env, capture_output=True, text=True)
finally:
    if askpass_path and os.path.exists(askpass_path): os.remove(askpass_path)
    git_env.pop('COLAB_GITHUB_TOKEN', None)
    token = None
PROJECT_DIR = REPO_DIR / 'BTL02'
os.chdir(PROJECT_DIR)
print('Checked out:', subprocess.check_output(['git', 'branch', '--show-current'], text=True).strip())


In [ ]:
import subprocess, sys
requirements = Path('requirements.txt').read_text(encoding='utf-8').splitlines()
filtered = [line for line in requirements if line.strip() and not line.lstrip().startswith('#') and line.split('=', 1)[0].strip().lower() not in {'torch', 'torchvision'}]
if sys.version_info >= (3, 13):
    filtered = ['scikit-learn==1.6.1' if line.strip().lower().startswith('scikit-learn==') else line for line in filtered]
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *filtered], check=True)
import torch, torchvision
if not torch.cuda.is_available(): raise RuntimeError('Bật GPU runtime Colab trước khi chạy huấn luyện.')
gpu_name = torch.cuda.get_device_name(0)
gpu_memory_gib = torch.cuda.get_device_properties(0).total_memory / 1024**3
print('PyTorch:', torch.__version__, '| Torchvision:', torchvision.__version__)
print('GPU:', gpu_name, '| VRAM GiB:', round(gpu_memory_gib, 1))
if not any(name in gpu_name.upper() for name in ('T4', 'A100')): raise RuntimeError(f'Notebook RQ2 yêu cầu runtime T4/A100; hiện tại là {gpu_name}.')
if gpu_memory_gib < 12: raise RuntimeError(f'VRAM không đủ an toàn cho ma trận này: {gpu_memory_gib:.1f} GiB; cần ít nhất 12 GiB.')


In [ ]:
import os, shutil
from google.colab import userdata
from src.data.downloader import FER2013Downloader
raw_dir = PROJECT_DIR / 'data' / 'raw'
raw_dir.mkdir(parents=True, exist_ok=True)
repo_csv = raw_dir / 'fer2013.csv'
drive_csv = DRIVE_ROOT / 'fer2013.csv'
if not repo_csv.is_file() and drive_csv.is_file():
    shutil.copy2(drive_csv, repo_csv)
if not repo_csv.is_file():
    kaggle_token = userdata.get('KAGGLE_API_TOKEN')
    if not kaggle_token: raise RuntimeError('Thiếu Colab Secret KAGGLE_API_TOKEN để tải FER2013.')
    os.environ['KAGGLE_API_TOKEN'] = kaggle_token
    del kaggle_token
    try:
        subprocess.run([sys.executable, '-m', 'src.data.downloader'], check=True, env=os.environ.copy())
    finally:
        os.environ.pop('KAGGLE_API_TOKEN', None)
split_counts = FER2013Downloader.validate_csv(repo_csv)
expected_splits = {'Training': 28709, 'PublicTest': 3589, 'PrivateTest': 3589}
assert split_counts == expected_splits, split_counts
if not drive_csv.is_file(): shutil.copy2(repo_csv, drive_csv)
print('FER2013 verified:', split_counts, '| Drive copy:', drive_csv)


In [ ]:
import os, shutil, subprocess, sys
from pathlib import Path
import pandas as pd
from src.train.mlflow_logger import MLflowLogger
from mlflow.tracking import MlflowClient

def link_drive_folder(name, target):
    link = PROJECT_DIR / 'outputs' / name
    link.parent.mkdir(parents=True, exist_ok=True)
    if link.is_symlink():
        if link.resolve() != target.resolve(): raise RuntimeError(f'outputs/{name} đã trỏ sang vị trí khác; dừng để bảo toàn dữ liệu.')
        return
    if link.exists():
        items = list(link.iterdir()) if link.is_dir() else []
        if link.is_dir() and all(item.name == '.gitkeep' and item.is_file() for item in items):
            for item in items: item.unlink()
            link.rmdir()
        else: raise RuntimeError(f'Không ghi đè outputs/{name} đang có dữ liệu; hãy sao lưu/di chuyển thủ công.')
    link.symlink_to(target, target_is_directory=True)

outputs = PROJECT_DIR / 'outputs'
outputs.mkdir(exist_ok=True)
link_drive_folder('checkpoints', CHECKPOINTS_DIR)
link_drive_folder('logs', LOGS_DIR)
link_drive_folder('figures', FIGURES_DIR)
os.environ['MLFLOW_TRACKING_URI'] = MLRUNS_DIR.as_uri()
os.environ['MLFLOW_ALLOW_FILE_STORE'] = 'true'
configs = [
    'experiments/sota/exp04_vgg16_finetune.yaml',
    'experiments/sota/exp05_resnet50_finetune.yaml',
    'experiments/sota/exp06_efficientnet_b0_finetune.yaml',
    'experiments/sota/exp04b_vgg16_scratch_128.yaml',
    'experiments/sota/exp05b_resnet50_scratch_128.yaml',
    'experiments/sota/exp06b_efficientnet_b0_scratch_128.yaml',
    'experiments/sota/exp04c_vgg16_pretrained_48.yaml',
    'experiments/sota/exp05c_resnet50_pretrained_48.yaml',
    'experiments/sota/exp06c_efficientnet_b0_pretrained_48.yaml',
]
import yaml
manifest = pd.DataFrame([yaml.safe_load(Path(path).read_text(encoding='utf-8')) for path in configs])
manifest = manifest[['exp_id', 'architecture', 'pretrained', 'image_size', 'rq_tag', 'seed', 'optimizer', 'lr', 'batch_size', 'epochs']]
manifest['run_name'] = manifest.apply(lambda row: f"{row['exp_id']}_{row['architecture']}_{'pretrained' if row['pretrained'] else 'scratch'}_{row['image_size']}_{row['optimizer']}", axis=1)
manifest_path = LOGS_DIR / 't0304_rq2_config_manifest.csv'
manifest.to_csv(manifest_path, index=False, encoding='utf-8-sig')
experiment_ids = {'EXP-04', 'EXP-05', 'EXP-06', 'EXP-04B', 'EXP-05B', 'EXP-06B', 'EXP-04C', 'EXP-05C', 'EXP-06C'}

def export_t0304_results(require_complete=False):
    logger = MLflowLogger()
    raw_path = LOGS_DIR / 't0304_all_mlflow_runs_24col.csv'
    logger.export_summary_csv(str(raw_path))
    all_runs = pd.read_csv(raw_path, encoding='utf-8-sig', keep_default_na=False)
    assert all_runs.columns.tolist() == MLflowLogger.CSV_COLUMNS
    selected = all_runs[all_runs['Name'].astype(str).isin(set(manifest['run_name']))].copy()
    run_to_exp = dict(zip(manifest['run_name'], manifest['exp_id']))
    selected['params.exp_id'] = selected['Name'].astype(str).map(run_to_exp)
    selected['_sort_time'] = pd.to_datetime(selected['Start Time'], errors='coerce', utc=True)
    selected = selected.sort_values('_sort_time').drop_duplicates(subset=['params.exp_id'], keep='last').drop(columns=['_sort_time'])
    result_path = DRIVE_ROOT / 't0304_rq2_transfer_vs_scratch_24col.csv'
    selected.reindex(columns=MLflowLogger.CSV_COLUMNS).to_csv(result_path, index=False, encoding='utf-8-sig')
    selected_manifest = manifest.merge(selected[['params.exp_id', 'Run ID', 'Status']], left_on='exp_id', right_on='params.exp_id', how='left').drop(columns=['params.exp_id'])
    selected_manifest.to_csv(LOGS_DIR / 't0304_rq2_run_manifest.csv', index=False, encoding='utf-8-sig')
    finished = set(selected.loc[selected['Status'].str.upper() == 'FINISHED', 'params.exp_id'])
    print(f'Finished {len(finished)}/9 configurations; incremental 24-column CSV:', result_path)
    if require_complete and finished != experiment_ids: raise RuntimeError(f'Chưa hoàn tất đủ 9 EXP IDs: {sorted(experiment_ids - finished)}')
    return selected, result_path

for config_path in configs:
    import yaml
    config = yaml.safe_load(Path(config_path).read_text(encoding='utf-8'))
    exp_id = config['exp_id']
    existing, _ = export_t0304_results()
    completed = existing[(existing['params.exp_id'] == exp_id) & (existing['Status'].str.upper() == 'FINISHED')]
    if not completed.empty:
        print(f'SKIP {exp_id}: FINISHED run already present in Drive MLflow store', flush=True)
        continue
    print(f'\n===== RUN {exp_id}: {config_path} =====', flush=True)
    subprocess.run([sys.executable, '-u', '-m', 'src.train.trainer', '--config', config_path], check=True, env=os.environ.copy())
    export_t0304_results()

results, results_csv = export_t0304_results(require_complete=True)
results_with_design = manifest.merge(results, left_on='run_name', right_on='Name', how='inner', suffixes=('_design', ''))
display(results_with_design[['exp_id', 'Name', 'Status', 'Run ID', 'architecture', 'pretrained', 'image_size', 'metrics.best_epoch', 'metrics.train_acc', 'metrics.best_val_acc', 'metrics.private_test_acc', 'metrics.macro_f1']])
print('Final 24-column CSV:', results_csv)
client = MlflowClient(tracking_uri=os.environ['MLFLOW_TRACKING_URI'])
epoch_times = []
detailed_metrics = []
for _, row in results.iterrows():
    history = client.get_metric_history(row['Run ID'], 'epoch_time_seconds')
    values = [point.value for point in history]
    if values: epoch_times.append({'exp_id': row['params.exp_id'], 'run_id': row['Run ID'], 'mean_epoch_seconds': sum(values) / len(values), 'epochs_logged': len(values)})
    run_data = client.get_run(row['Run ID']).data
    run_metrics, run_params = run_data.metrics, run_data.params
    detailed_metrics.append({'exp_id': row['params.exp_id'], 'run_id': row['Run ID'], 'architecture': run_params.get('architecture'), 'pretrained': run_params.get('pretrained'), 'image_size': run_params.get('image_size'), 'total_params': run_params.get('total_params'), 'best_epoch': run_metrics.get('best_epoch'), 'train_acc': run_metrics.get('train_acc'), 'public_test_acc': run_metrics.get('best_val_acc'), 'public_macro_f1': run_metrics.get('best_val_macro_f1'), 'private_test_acc': run_metrics.get('private_test_acc'), 'private_macro_f1': run_metrics.get('macro_f1'), 'mean_epoch_seconds': sum(values) / len(values) if values else None})
epoch_times_df = pd.DataFrame(epoch_times)
epoch_times_path = LOGS_DIR / 't0304_epoch_times.csv'
epoch_times_df.to_csv(epoch_times_path, index=False)
print('Per-run epoch timing:', epoch_times_path)
detailed_path = LOGS_DIR / 't0304_rq2_detailed_metrics.csv'
pd.DataFrame(detailed_metrics).to_csv(detailed_path, index=False, encoding='utf-8-sig')
print('Detailed RQ2 metrics (PublicTest validation, PrivateTest, Macro-F1, params, timing):', detailed_path)


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
results = pd.read_csv(DRIVE_ROOT / 't0304_rq2_transfer_vs_scratch_24col.csv', encoding='utf-8-sig', keep_default_na=False)
manifest = pd.read_csv(LOGS_DIR / 't0304_rq2_config_manifest.csv', encoding='utf-8-sig')
assert len(results.columns) == 24 and set(results['Status'].str.upper()) == {'FINISHED'}
results = manifest.merge(results, left_on='run_name', right_on='Name', how='inner', suffixes=('_design', ''))
results['label'] = results['architecture'].astype(str) + ' | pretrained=' + results['pretrained'].astype(str) + ' | ' + results['image_size'].astype(str) + 'px'
fig, axes = plt.subplots(1, 2, figsize=(17, 6))
for column, title, axis in [('metrics.best_val_acc', 'PublicTest validation accuracy (%)', axes[0]), ('metrics.private_test_acc', 'PrivateTest accuracy (%)', axes[1])]:
    values = pd.to_numeric(results[column], errors='coerce')
    axis.bar(results['label'], values)
    axis.set_title(title); axis.set_ylabel('Accuracy (%)'); axis.tick_params(axis='x', rotation=75)
    axis.axhline(69 if column == 'metrics.best_val_acc' else 71, color='red', linestyle='--', label='project reference (69/71%)')
    axis.legend()
fig.tight_layout()
figure_path = FIGURES_DIR / 't0304_accuracy_comparison.png'
fig.savefig(figure_path, dpi=160, bbox_inches='tight')
print('Comparison figure:', figure_path)
print('Note: best_val_acc is the PublicTest metric used by the existing Trainer for checkpoint selection; the 69/71 lines are references, not measured results.')
